# Forecast-period surface energy budget

Computes reusable member-level surface-energy-budget caches and plots regional ensemble summaries. Increase `max_members` after a one-member smoke test.


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")

workflow_name = "fcst_atm_model_evaluation"
run_segment = "fc"
season = "S0"
experiments = ("CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0")
max_members = 1
plot_variables = ("Rn", "LE", "SH", "Residual", "BowenRatio")
region = "land"
force_compute = False
show = True

In [ ]:
import sys
import matplotlib.pyplot as plt
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.bias_metrics import experiment_registry
from util.atm_surface_energy_budget import INPUT_VARIABLES, OUTPUT_VARIABLES, REGIONS, energy_budget_cache_path, cache_surface_energy_budget, plot_energy_budget_timeseries


## Notes: available variables and regions

Required model inputs: `LHFLX` (or `QFLX`, converted using latent heat of vaporization), `SHFLX`, `FSNS`, and `FLNS`. Cached derived variables: `Rn = FSNS - FLNS`, `LE`, `SH`, `Residual = Rn - (LE + SH)`, and `BowenRatio = SH / LE`.

Plot regions: `global`, `land`, and `ocean`. Existing member caches are reused unless `force_compute = True`. Analysis notebooks prefer `6hourly_da`, while forecast notebooks prefer daily files and fall back to 6-hourly data.


In [ ]:
data_dir, figure_dir = workflow_output_dirs(workflow_name, "surface_energy_budget", output_root=DIAGNOSTIC_OUTPUT_ROOT)
registry = experiment_registry(INPUT_DATA_ROOT, season, run_segment)
unknown = set(experiments) - set(registry)
if unknown:
    raise ValueError(f"Unavailable experiments: {sorted(unknown)}; choose from {list(registry)}")
landmask = INPUT_DATA_ROOT / "reference/lnd_sea_mask/landmask_1x1.nc"
processed = {}
for experiment in experiments:
    meta = registry[experiment]
    count = meta["nens"] if max_members is None else min(max_members, meta["nens"])
    members = {}
    for number in range(1, count + 1):
        member = f"EN{number:02d}"
        target = energy_budget_cache_path(data_dir, experiment, member, meta["period"])
        existed = target.is_file()
        members[member] = cache_surface_energy_budget(target, data_root=INPUT_DATA_ROOT, experiment=experiment, season=season, run_segment=run_segment, member=member, period=meta["period"], force_compute=force_compute)
        print(("[REUSE]" if existed and not force_compute else "[WRITE]"), target)
    processed[experiment] = members
for variable in plot_variables:
    fig = plot_energy_budget_timeseries(processed, variable, region, landmask, label_map={name:registry[name]["label"] for name in experiments})
    output = figure_dir / f"{variable}_{region}_{season}_{run_segment}.png"
    fig.savefig(output, dpi=200, bbox_inches="tight")
    if not show: plt.close(fig)
    print(f"[FIGURE] {output}")
